# AI Agent 是怎么一步步长出来的？

这份笔记按**时间顺序**讲一件事：人们怎么从「只会聊天的模型」，一路走到「能接工具、能接系统、能工程化落地」的 Agent。

**在 Cloud Studio 上**：先运行下一格 `!pip install` 装好依赖；直接调用填好的 **DeepSeek API Key**。天气仍用 **Open-Meteo**，无需天气密钥。若将笔记本公开，务必删掉密钥。

**你需要会什么**：会点「Run All」或按顺序运行单元格即可，如果需要自己构建属于自己的Agent，还需要会写下函数。

# 培训目的
1、目的是建立对Agent的数据交互直觉，知道所有的架构比方Harness、Skill、MCP、Function call是具体如何和大模型进行交互的。

2、不管说什么开源框架，比方近期很火的Hermes和之前很火的Openclaw，都是基于下文所说的基本单元概念构建的，希望大家在看了这篇文档后，都能学会根据自己的需求构建自己的Agent，以及不管未来媒体鼓吹什么概念，能够理解他本质上是啥。‘

    **大模型除了prompt和output以外，没有任何东西了**

[def]: image.png

In [ ]:
# Cloud Studio：依赖与 DeepSeek 密钥（请把引号内换成你的 sk-）
# 只安装 OpenAI 兼容客户端和 HTTP 请求库；后面示例不使用任何 Agent 框架。
!pip install -q openai requests


In [1]:
DEEPSEEK_API_KEY = "sk-dfc6d177f3be4b2ca4a3a2da1a72f8f5"  # 例如: "sk-xxxxxxxx"


## 发展时间线一览

1. **调大模型 API**：程序像把一段话发给云端模型，拿回一段文字。
2. **Function Call（工具调用）**：人们并不满足模型只能输出文本，需求模型输出结构化的「调哪个函数、参数是什么」——程序负责真正执行。
3. **MCP**：**一句话**：标准化的Function Call——把 Function Call 的接法（发现工具、传参、回传结果）做成**标准协议**，方便不同主题与工具服务互接；能力上还是「会调用工具」，没有新魔法。
4. **Skill**：**一句话**：本质上是会选择调用的prompt模板———把「这类活该怎么干」写成**可复用说明书**（何时用、步骤、约束、输出风格），通常塞进 **system** 或独立文档，让模型行为**稳定、可复制**。
5. **Harness Engineer（驾驭工程）**：提示词、工具、记忆、评测、监控、安全——**成套工程**装起来才能稳定上线。

由时间线看，是从「会说话」———（API调用）到「会办事」———（Function Call），再到「标准对接」——（MCP）和「经验沉淀」——(Skill)，最后到「能做交付的系统」——（Harness Engineer）。

由工程上来说是从Prompt Engineer（提示词工程）—— Context Engineer（上下文工程）——Harness Engineer（驾驭工程），本质上还是围绕提示词做文章，所以我们说前面说**大模型除了prompt和output以外，没有任何东西了**，下一步我们会在实际代码中具体体现。

从流程上来讲是最终是需要实现从需求——Agent——自主交付。

大家可以想一下，大模型像燃料，塞进了名叫Function Call的气缸里——它第一次开始和真实世界开始交互，开始做功，用Skill把多个气缸组成了一个发动机，构成了汽车最关键的零件，光有发动机是不够的，你需要轮胎，转向器，刹车，油门，方向盘等等等等，才能真正构建一辆汽车，这些零配件，就是Harness Engineer。

下面我们按这个顺序，用**几段短代码**把关键几步跑通。


---
## 第一阶段：调用大模型 API（约2020 年前后普及的思路）

**一句话**：你的程序构造一段「提示」，HTTP 请求发给服务商，返回模型的生成文本。

**最小闭环**：发一条 → 收一条。先不关心工具、不关心 Agent。下面使用 **DeepSeek**兼容 OpenAI 的接口做真实调用。


In [ ]:
# 真实调用 DeepSeek Chat API（密钥见上一格 DEEPSEEK_API_KEY）
from openai import OpenAI

if not DEEPSEEK_API_KEY.strip():
    raise RuntimeError("请先在上一格填写 DEEPSEEK_API_KEY")

client = OpenAI(api_key=DEEPSEEK_API_KEY.strip(), base_url="https://api.deepseek.com")

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": "用一句话解释什么是 API"}],
)

print(response.choices[0].message.content)


**和真实 API 的对应关系（记这个就够）：**

- 你发送：`messages`（多轮对话）、可选 `model` 名称等。
- 你收到：通常至少有一段 `content`（模型说的话）。

到这一步，AI 还只是一个**文本生成器**，不会替你查天气、不会替你点数据库——除非你手写一堆 `if` 去解析文字（又慢又脆）。所以需要下一阶段。

---
## 第二阶段：Function Call / Tool Use（工具调用，约 2023 起各厂商主推）

**一句话**：模型除了返回「给人看的句子」，还可以返回**结构化的「我要调用哪个工具、参数是什么」**。你的程序读到这个结构，去执行真正的函数（查表、算数、调内部系统），再把结果塞回对话里。

**最小闭环**：
1. 声明一个工具（名字 + 参数说明，交给 API 的 `tools` 参数）。
2. 模型真实返回 `tool_calls`（结构化「要调谁、参数是什么」）。
3. 你的代码执行工具（下面用 **Open-Meteo** 查天气，免密钥），把结果用 `role=tool` 发回模型，再取最终自然语言回答。
4. **补充**：`tool_calls` 里的 `city` 可能有多种格式。下面在请求地理编码前**再调一次大模型**：把入参规则写在 `system`（`_CITY_PARAM_NORMALIZE_SYSTEM`）里，让它输出**单行规范化地名**，不维护硬编码城市列表。这是最初步的Function Call的入参模式，像磐心里的Chat Ops就是使用该模式，后面大模型厂商都做了原生配置即Json Schema，可以使得模型直接输出符合规则的Json格式。

这样就不用在自然语言里猜「它是不是想查天气」——**协议层**解决意图；**规范化层**把规则交给模型，由它产出可用的 `city`。

**另外还有一种很适合高性能模型的Function Call方式，就是只定义执行终端——命令行（CLI）,比方Java、Python或者Bash，让模型自己写代码运行，这样更灵活，但是对模型的要求更高。**

#感知——思考——行动——观察——循环至模型不再执行Function Call，总结并输出。
![流程](./image.png)

In [ ]:
# Function Call 最小闭环：模型把自然语言日期需求拆成结构化查询，Python 负责真实计算。
import json
from datetime import date, timedelta
from openai import OpenAI

if not DEEPSEEK_API_KEY.strip():
    raise RuntimeError("请先在上一格填写 DEEPSEEK_API_KEY")

client = OpenAI(api_key=DEEPSEEK_API_KEY.strip(), base_url="https://api.deepseek.com")

# 工具：根据“相对今天的天数”计算日期。模型负责理解“昨天/三天后/下周一”等自然语言。
def get_date_info(queries):
    weekday_names = ["周一", "周二", "周三", "周四", "周五", "周六", "周日"]
    today = date.today()
    result = []

    for item in queries:
        d = today + timedelta(days=item["offset_days"])
        result.append({
            "label": item["label"],
            "date": d.isoformat(),
            "weekday": weekday_names[d.weekday()],
            "need_date": item.get("need_date", True),
            "need_weekday": item.get("need_weekday", True),
        })
    return {"today": today.isoformat(), "items": result}

# schema 让模型输出任意相对日期 offset_days。
tools = [{
    "type": "function",
    "function": {
        "name": "get_date_info",
        "description": "按相对今天的天数查询日期和星期。today 的 offset_days=0，昨天=-1，明天=1，三天后=3。",
        "parameters": {
            "type": "object",
            "properties": {
                "queries": {
                    "type": "array",
                    "description": "用户问题里每一个日期需求，按原表达拆成一项。",
                    "items": {
                        "type": "object",
                        "properties": {
                            "label": {"type": "string", "description": "用户原始表达，如 昨天、今天、三天后、下周一。"},
                            "offset_days": {"type": "integer", "description": "这个日期相对今天的天数。"},
                            "need_date": {"type": "boolean", "description": "用户是否关心几月几号。"},
                            "need_weekday": {"type": "boolean", "description": "用户是否关心周几。"},
                        },
                        "required": ["label", "offset_days", "need_date", "need_weekday"],
                        "additionalProperties": False,
                    },
                }
            },
            "required": ["queries"],
            "additionalProperties": False,
        },
    },
}]

system_prompt = f"今天的真实日期是 {date.today().isoformat()}。请把用户的日期/星期需求转换成 get_date_info 的 offset_days 参数。"
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "昨天是周几，是几号？"},
]

print("[1] system prompt：把真实今天告诉模型，方便它理解相对日期")
print(system_prompt)
print("\n[2] 用户 prompt:")
print(messages[-1]["content"])
print("\n[3] 可用工具 schema:")
print(json.dumps(tools, ensure_ascii=False, indent=2))

print("\n[4] 第一次调用模型：让模型把自然语言日期需求转成工具参数 ...")
first_request = {
    "model": "deepseek-chat",
    "messages": messages,
    "tools": tools,
    "tool_choice": "auto",
}
print("第一次调用发给大模型的完整输入:")
print(json.dumps(first_request, ensure_ascii=False, indent=2))

first = client.chat.completions.create(**first_request)
assistant_msg = first.choices[0].message

print("\n第一次调用大模型的完整输出:")
print(first.model_dump_json(indent=2))
print("\n模型返回的 tool_calls:")
print(assistant_msg.tool_calls)

if assistant_msg.tool_calls:
    # API 返回的是对象；转成普通 dict 后再放回 messages，便于初学者理解。
    messages.append(assistant_msg.model_dump(exclude_none=True))

    for tool_call in assistant_msg.tool_calls:
        args = json.loads(tool_call.function.arguments)
        print("\n[5] Python 执行工具:", tool_call.function.name)
        print("工具入参:")
        print(json.dumps(args, ensure_ascii=False, indent=2))

        if tool_call.function.name == "get_date_info":
            tool_result = get_date_info(**args)
        else:
            tool_result = {"error": f"未知工具: {tool_call.function.name}"}

        print("工具结果:")
        print(json.dumps(tool_result, ensure_ascii=False, indent=2))
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(tool_result, ensure_ascii=False),
        })

    print("\n[6] 把工具结果放回 messages，再次调用模型生成自然语言回答 ...")
    second_request = {
        "model": "deepseek-chat",
        "messages": messages,
    }
    print("第二次调用发给大模型的完整输入:")
    print(json.dumps(second_request, ensure_ascii=False, indent=2))

    final = client.chat.completions.create(**second_request)
    print("\n第二次调用大模型的完整输出:")
    print(final.model_dump_json(indent=2))

    print("\n[7] 最终回答:")
    print(final.choices[0].message.content)
else:
    print("\n模型没有调用工具，直接回答:")
    print(assistant_msg.content)



**零基础要记住的一句：**  
Function Call = **模型输出「动作指令」+ 你的程序负责「真的动手」**。  
没有这一步，就很难稳定地做「会办事」的助手。

---
## 第三阶段：MCP（约 2024 起热起来）——**简单带过**

**一句话**：MCP（Model Context Protocol）本质上就是**标准化的 Function Call 接线**：工具怎么发现、怎么传参、结果怎么回传，都按同一套协议来。你已经会 `tools` / `tool_calls` / `role=tool` 了；MCP 解决的是**跨产品复用**时少写胶水代码。

**本课不写 MCP 的具体输出，因为和Function Call没区别**：思想上把它当成「**工程上的统一接口**」，与下一节的 **Skill（做事章法）** 不是一回事——后者解决的是**任务怎么说清楚**即Prompt的工作，不是Function Call的工作。


In [ ]:
# MCP 极简模拟：一个按日期查天气的工具，过去走 Archive，今天/未来短期走 Forecast。
import json
import requests
from datetime import date as Date
from openai import OpenAI

if not DEEPSEEK_API_KEY.strip():
    raise RuntimeError("请先在上一格填写 DEEPSEEK_API_KEY")

client = OpenAI(api_key=DEEPSEEK_API_KEY.strip(), base_url="https://api.deepseek.com")

# JSON Schema 怎么写：
# 1. type="object"：工具入参整体是一个 JSON 对象，也就是 Python dict。
# 2. properties：逐个声明字段名、字段类型、字段含义。模型会参考 description 填参数。
# 3. date 用 YYYY-MM-DD 字符串：模型把“昨天/明天/下周三”先理解成标准日期，再传给工具。
# 4. required：city 和 date 必须出现；否则工具不知道查哪里、查哪天。
# 5. additionalProperties=False：不允许模型临时发明多余字段，减少脏参数。
# 6. MCP 中通常叫 inputSchema；OpenAI 兼容接口中同一块 schema 放在 parameters。
weather_input_schema = {
    "type": "object",
    "properties": {
        "city": {"type": "string", "description": "要查询天气的城市名，例如 苏州、北京、Shanghai。"},
        "date": {"type": "string", "description": "要查询的日期，格式必须是 YYYY-MM-DD。"},
    },
    "required": ["city", "date"],
    "additionalProperties": False,
}

# 真实 MCP 工具描述会由 MCP Server 返回；这里直接写成字典，方便看清本质。
mcp_weather_tool = {
    "name": "get_weather_by_date",
    "description": "按城市和日期查询天气。过去日期走 Open-Meteo Archive，今天和未来短期走 Forecast。",
    "inputSchema": weather_input_schema,
}

# OpenAI 兼容接口的 tools 格式：把 MCP 的 inputSchema 放进 parameters。
openai_tools = [{
    "type": "function",
    "function": {
        "name": mcp_weather_tool["name"],
        "description": mcp_weather_tool["description"],
        "parameters": mcp_weather_tool["inputSchema"],
    },
}]

def get_weather_by_date(city, date):
    # 工具真正和环境交互：先用城市名查经纬度，再按日期选择历史天气或天气预报接口。
    target_date = Date.fromisoformat(date)
    today = Date.today()

    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1, "language": "zh", "format": "json"},
        timeout=15,
    ).json()
    results = geo.get("results") or []
    if not results:
        return {"error": f"没找到城市：{city}"}

    place = results[0]
    is_past = target_date < today
    url = "https://archive-api.open-meteo.com/v1/archive" if is_past else "https://api.open-meteo.com/v1/forecast"
    source = "archive历史天气" if is_past else "forecast预报天气"

    data = requests.get(
        url,
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "start_date": date,
            "end_date": date,
            "daily": "temperature_2m_max,temperature_2m_min,weather_code,wind_speed_10m_max",
            "timezone": "auto",
        },
        timeout=15,
    ).json()

    if "daily" not in data:
        return {"error": "天气接口没有返回 daily 数据", "raw": data}

    daily = data["daily"]
    return {
        "city": place.get("name", city),
        "country": place.get("country", ""),
        "date": date,
        "source": source,
        "temperature_2m_min": daily["temperature_2m_min"][0],
        "temperature_2m_max": daily["temperature_2m_max"][0],
        "weather_code": daily["weather_code"][0],
        "wind_speed_10m_max": daily["wind_speed_10m_max"][0],
    }

system_prompt = f"今天的真实日期是 {Date.today().isoformat()}。请把用户说的昨天、明天、下周等时间转换成 YYYY-MM-DD。"
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "请通过工具查一下苏州明天天气怎么样，并给一句出门建议。"},
]

print("[1] MCP 工具描述，也就是工具服务对外暴露的能力:")
print(json.dumps(mcp_weather_tool, ensure_ascii=False, indent=2))
print("\n[2] 转成大模型接口需要的 tools 格式:")
print(json.dumps(openai_tools, ensure_ascii=False, indent=2))

print("\n[3] 第一次调用模型：让模型根据用户问题和 JSON Schema 生成工具入参。")
first_request = {
    "model": "deepseek-chat",
    "messages": messages,
    "tools": openai_tools,
    "tool_choice": "auto",
}
print("第一次发给大模型的完整输入:")
print(json.dumps(first_request, ensure_ascii=False, indent=2))

first = client.chat.completions.create(**first_request)
assistant_msg = first.choices[0].message
print("\n第一次大模型的完整输出:")
print(json.dumps(first.model_dump(), ensure_ascii=False, indent=2))

if assistant_msg.tool_calls:
    # 把模型输出的 tool_calls 也作为历史消息放回去，下一轮模型才知道自己刚才要求调过工具。
    messages.append(assistant_msg.model_dump(exclude_none=True))

    for tool_call in assistant_msg.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        print("\n[4] Python 按模型给出的参数执行真实天气工具:")
        print("工具名:", tool_call.function.name)
        print("工具入参:")
        print(json.dumps(arguments, ensure_ascii=False, indent=2))

        tool_result = get_weather_by_date(**arguments)
        print("工具结果:")
        print(json.dumps(tool_result, ensure_ascii=False, indent=2))

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(tool_result, ensure_ascii=False),
        })

    print("\n[5] 第二次调用模型：把工具结果放回上下文，让模型组织成人话。")
    second_request = {
        "model": "deepseek-chat",
        "messages": messages,
    }
    print("第二次发给大模型的完整输入:")
    print(json.dumps(second_request, ensure_ascii=False, indent=2))

    final = client.chat.completions.create(**second_request)
    print("\n第二次大模型的完整输出:")
    print(json.dumps(final.model_dump(), ensure_ascii=False, indent=2))

    print("\n[6] 最终回答:")
    print(final.choices[0].message.content)
else:
    print("\n模型没有调用工具，完整输出里会看到 content 而不是 tool_calls:")
    print(assistant_msg.content)



---
## 第四阶段：Skill（技能包）——**本课重点**

**一句话**：Skill 是「**这类活该怎么干**」的说明书。模型先读章法，再调用工具、组织回答，比「只给工具、不给规矩」稳定得多。

**一份 Skill 里建议至少包含这些块**（下面代码里用字典字段一一对应）：

1. **名称 / 定位**（`name`、`summary`）：这条技能叫什么、解决哪类任务。  
2. **何时使用**（`when_to_use`）：什么问题必须走这条流程，减少乱用。  
3. **步骤**（`steps`）：先后顺序——例如必须先调工具、再用一句话总结。  
4. **约束**（`constraints`）：禁止编造未返回的数据、单位、失败时怎么说等。  
5. **输出风格**（`output_style`）：口语还是书面、长短、是否分点。  
6. **（可选）示例**：好/坏样例；本课为缩短篇幅略去，培训时可口头补充。

**怎么落地**：把上述内容拼成一段 **`role=system` 的提示**，再和 **Function Call** 叠在一起用——**Skill 管章法，工具管动手能力**。


In [14]:
# Skill + 多工具调用：Skill 负责出行提醒章法，日期和天气插件负责拿真实信息。
import json
import requests
from datetime import date, timedelta
from openai import OpenAI

if not DEEPSEEK_API_KEY.strip():
    raise RuntimeError("请先在上一格填写 DEEPSEEK_API_KEY")

client = OpenAI(api_key=DEEPSEEK_API_KEY.strip(), base_url="https://api.deepseek.com")

# 一个 Skill 至少要把这些基本元素说清楚：什么时候用、按什么步骤做、不能做什么、怎么输出。
SKILL = {
    "name": "dated_travel_brief_skill",
    "summary": "把目标日期、星期和目标城市的指定日期天气合在一起，给用户一条出行提醒。",
    "when_to_use": "当用户询问昨天、今天、明天、三天后等某个日期去某城市出行、出差、游玩是否方便时使用。",
    "steps": [
        "先识别用户提到的目标日期和目标城市。",
        "调用 get_date_info 查询目标日期的真实日期和星期。",
        "调用 get_weather_by_date 查询该城市在目标日期的天气。",
        "综合两个工具结果，给出一句清楚、实用的出行建议。",
    ],
    "constraints": [
        "不要凭记忆猜日期或天气。",
        "天气工具支持历史天气和短期预报，但不代表能查任意遥远未来。",
        "天气结果来自工具返回，不能编造工具没有返回的数据。",
        "回答时说明天气数据是历史天气还是预报天气。",
    ],
    "output_style": "中文、简洁、口语化，优先一句话；必要时最多两句话。",
    "example": "用户问：明天去苏州出差要注意什么？回答：明天是周日，苏州预报最高约18℃，建议带薄外套。",
}

def skill_to_system_prompt(skill):
    # Skill 本质上会被塞进 system prompt，告诉模型“这类任务该怎么做”。
    return f"""
你正在使用一个 Skill，请严格按它做事。
名称：{skill['name']}
定位：{skill['summary']}
何时使用：{skill['when_to_use']}
步骤：{'; '.join(skill['steps'])}
约束：{'; '.join(skill['constraints'])}
输出风格：{skill['output_style']}
示例：{skill['example']}
""".strip()

def get_date_info(queries):
    # 日期插件：模型负责把“明天/三天后”理解成 offset_days，Python 负责算真实日期。
    weekday_names = ["周一", "周二", "周三", "周四", "周五", "周六", "周日"]
    today = date.today()
    items = []
    for query in queries:
        d = today + timedelta(days=query["offset_days"])
        items.append({
            "label": query["label"],
            "date": d.isoformat(),
            "weekday": weekday_names[d.weekday()],
        })
    return {"today": today.isoformat(), "items": items}

def get_weather_by_date(city, target_date):
    # 天气插件：先把城市名换成经纬度，再按日期选择历史天气 Archive 或预报天气 Forecast。
    query_date = date.fromisoformat(target_date)
    today = date.today()

    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1, "language": "zh", "format": "json"},
        timeout=15,
    ).json()
    results = geo.get("results") or []
    if not results:
        return {"error": f"没找到城市：{city}"}

    place = results[0]
    is_past = query_date < today
    url = "https://archive-api.open-meteo.com/v1/archive" if is_past else "https://api.open-meteo.com/v1/forecast"
    source = "archive历史天气" if is_past else "forecast预报天气"

    data = requests.get(
        url,
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "start_date": target_date,
            "end_date": target_date,
            "daily": "temperature_2m_max,temperature_2m_min,weather_code,wind_speed_10m_max",
            "timezone": "auto",
        },
        timeout=15,
    ).json()
    if "daily" not in data:
        return {"error": "天气接口没有返回 daily 数据", "raw": data}

    daily = data["daily"]
    return {
        "city": place.get("name", city),
        "country": place.get("country", ""),
        "date": target_date,
        "source": source,
        "temperature_2m_min": daily["temperature_2m_min"][0],
        "temperature_2m_max": daily["temperature_2m_max"][0],
        "weather_code": daily["weather_code"][0],
        "wind_speed_10m_max": daily["wind_speed_10m_max"][0],
    }

# 给模型的两个插件说明：模型只看 schema，不会直接运行 Python；真正运行在下面的 if 分发里完成。
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_date_info",
            "description": "按相对今天的天数查询日期和星期。今天=0，明天=1，昨天=-1，三天后=3。",
            "parameters": {
                "type": "object",
                "properties": {
                    "queries": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "label": {"type": "string", "description": "用户原始日期表达，如 明天、三天后。"},
                                "offset_days": {"type": "integer", "description": "这个日期相对今天的天数。"},
                            },
                            "required": ["label", "offset_days"],
                            "additionalProperties": False,
                        },
                    }
                },
                "required": ["queries"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather_by_date",
            "description": "按城市和日期查询天气。过去日期走 Open-Meteo Archive，今天和未来短期走 Forecast。",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "要查询天气的城市名，例如 苏州、北京、Shanghai。"},
                    "target_date": {"type": "string", "description": "要查询的日期，格式必须是 YYYY-MM-DD。"},
                },
                "required": ["city", "target_date"],
                "additionalProperties": False,
            },
        },
    },
]

system_prompt = skill_to_system_prompt(SKILL) + f"\n\n今天的真实日期是 {date.today().isoformat()}。"
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "我明天去苏州出差，帮我看下明天是周几、苏州明天天气怎么样，并给一句出门建议。"},
]

print("[1] Skill 原始结构:")
print(json.dumps(SKILL, ensure_ascii=False, indent=2))
print("\n[2] Skill 转成 system prompt:")
print(system_prompt)
print("\n[3] 可用插件 schema:")
print(json.dumps(tools, ensure_ascii=False, indent=2))

print("\n[4] 第一次调用模型：让模型根据 Skill 决定要调用哪些插件。")
first_request = {
    "model": "deepseek-chat",
    "messages": messages,
    "tools": tools,
    "tool_choice": "auto",
}
print("第一次发给大模型的完整输入:")
print(json.dumps(first_request, ensure_ascii=False, indent=2))

first = client.chat.completions.create(**first_request)
assistant_msg = first.choices[0].message
print("\n第一次大模型的完整输出:")
print(json.dumps(first.model_dump(), ensure_ascii=False, indent=2))

if assistant_msg.tool_calls:
    # 这条 assistant 消息包含 tool_calls，要放回 messages，模型下一轮才知道自己刚才要求调了哪些工具。
    messages.append(assistant_msg.model_dump(exclude_none=True))

    for tool_call in assistant_msg.tool_calls:
        args = json.loads(tool_call.function.arguments)
        print("\n[5] Python 执行插件:", tool_call.function.name)
        print("插件入参:")
        print(json.dumps(args, ensure_ascii=False, indent=2))

        if tool_call.function.name == "get_date_info":
            tool_result = get_date_info(**args)
        elif tool_call.function.name == "get_weather_by_date":
            tool_result = get_weather_by_date(**args)
        else:
            tool_result = {"error": f"未知插件: {tool_call.function.name}"}

        print("插件结果:")
        print(json.dumps(tool_result, ensure_ascii=False, indent=2))
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(tool_result, ensure_ascii=False),
        })

    print("\n[6] 第二次调用模型：把所有插件结果回填，让模型按 Skill 风格组织回答。")
    second_request = {
        "model": "deepseek-chat",
        "messages": messages,
    }
    print("第二次发给大模型的完整输入:")
    print(json.dumps(second_request, ensure_ascii=False, indent=2))

    final = client.chat.completions.create(**second_request)
    print("\n第二次大模型的完整输出:")
    print(json.dumps(final.model_dump(), ensure_ascii=False, indent=2))

    print("\n[7] 最终回答:")
    print(final.choices[0].message.content)
else:
    print("\n模型没有调用插件，完整输出里会看到 content 而不是 tool_calls:")
    print(assistant_msg.content)



[1] Skill 原始结构:
{
  "name": "dated_travel_brief_skill",
  "summary": "把目标日期、星期和目标城市的指定日期天气合在一起，给用户一条出行提醒。",
  "when_to_use": "当用户询问昨天、今天、明天、三天后等某个日期去某城市出行、出差、游玩是否方便时使用。",
  "steps": [
    "先识别用户提到的目标日期和目标城市。",
    "调用 get_date_info 查询目标日期的真实日期和星期。",
    "调用 get_weather_by_date 查询该城市在目标日期的天气。",
    "综合两个工具结果，给出一句清楚、实用的出行建议。"
  ],
  "constraints": [
    "不要凭记忆猜日期或天气。",
    "天气工具支持历史天气和短期预报，但不代表能查任意遥远未来。",
    "天气结果来自工具返回，不能编造工具没有返回的数据。",
    "回答时说明天气数据是历史天气还是预报天气。"
  ],
  "output_style": "中文、简洁、口语化，优先一句话；必要时最多两句话。",
  "example": "用户问：明天去苏州出差要注意什么？回答：明天是周日，苏州预报最高约18℃，建议带薄外套。"
}

[2] Skill 转成 system prompt:
你正在使用一个 Skill，请严格按它做事。
名称：dated_travel_brief_skill
定位：把目标日期、星期和目标城市的指定日期天气合在一起，给用户一条出行提醒。
何时使用：当用户询问昨天、今天、明天、三天后等某个日期去某城市出行、出差、游玩是否方便时使用。
步骤：先识别用户提到的目标日期和目标城市。; 调用 get_date_info 查询目标日期的真实日期和星期。; 调用 get_weather_by_date 查询该城市在目标日期的天气。; 综合两个工具结果，给出一句清楚、实用的出行建议。
约束：不要凭记忆猜日期或天气。; 天气工具支持历史天气和短期预报，但不代表能查任意遥远未来。; 天气结果来自工具返回，不能编造工具没有返回的数据。; 回答时说明天气数据是历史天气还是预报天气。
输出风格：中文、简洁、口语化，优先一句话；必要时最多两句话。
示例

---
## 第五阶段：Harness Engineer（编排工程师 / Agent 工程化）

**一句话**：前面几样都是「零件」。真正上线时，你要搭**整套缰绳与马鞍（harness）**：

- **提示词与流程**：什么时候规划、什么时候调用工具、什么时候停。
- **记忆与上下文**：哪些该记住、哪些该截断、隐私怎么控。
- **评测**：哪些任务算成功，回归测试怎么做。
- **可观测与安全**：日志、限流、权限、防注入、人工兜底。

**Harness Engineer** 不是只会调 API的人，而是把 Agent **当产品/系统**来做的人：能设计闭环、能排障、能量化效果。

**最小闭环**：问自己四个问题就够入门—— 
1. 输入输出边界清不清楚？  
2. 工具失败时怎么办？  
3. 怎么证明变好了（指标）？  
4. 出了问题能不能快速定位（日志/追踪）？

**PS:关于Harness，大家不用过于担心，是在于你之前的任务遇到问题了，无法实现你的需求了，所做的一些工程设计。等遇到问题再考虑好了，比方你工具执行报错后怎么排障？多轮对话上下文太长了导致Agent执行效果不好，上下文怎么压缩？没有执行到你满意的方向怎么控制执行方向？涉及CLI工具担心做一些有风险的执行怎么限制？等遇到问题，再用Harness解决问题就好。**

![harness](./image3.png)

In [ ]:
# Harness 最小工程化闭环：把上一节 Skill + 工具调用包成“可拦截、可治理、可追踪”的 Agent 系统。
# 主题任务：出差提醒（解析自然语言 -> 调日期/天气工具 -> 校验和治理输出 -> 留下可排障追踪）

import json
import logging
import re
import sys
from dataclasses import dataclass, field
from datetime import date, timedelta
from typing import Any, Callable

import requests
from openai import OpenAI

if not DEEPSEEK_API_KEY.strip():
    raise RuntimeError("请先在上一格填写 DEEPSEEK_API_KEY")

client = OpenAI(api_key=DEEPSEEK_API_KEY.strip(), base_url="https://api.deepseek.com")


# ---------- 1) 上下文管理 ----------
# 作用：Harness 负责维护要塞给 Agent 的 messages，包括 system、user、assistant，以及验证失败后的反馈信息。
# 这里演示一个最小策略：超过 max_chars 后丢弃较早对话，避免上下文过长、成本失控、旧信息干扰新任务。
class ContextManager:
    def __init__(self, max_chars: int = 3000) -> None:
        self.max_chars = max_chars
        self._messages: list[dict[str, str]] = []

    def reset_with_system(self, system: str) -> None:
        self._messages = [{"role": "system", "content": system}]

    def append(self, role: str, content: str) -> None:
        self._messages.append({"role": role, "content": content})
        self._trim_if_needed()

    @property
    def messages(self) -> list[dict[str, str]]:
        return list(self._messages)

    def _trim_if_needed(self) -> None:
        total = sum(len(m["content"]) for m in self._messages)
        while total > self.max_chars and len(self._messages) > 2:
            removed = self._messages.pop(1)
            total -= len(removed["content"])


# ---------- 2) 状态持久化 ----------
# 作用：保存跨轮仍然要记住的运行状态，例如上次工具结果、步骤计数、用户偏好、任务进度。
# 它和上下文管理不同：上下文管理是“这轮给模型看什么”，状态持久化是“系统自己长期记住什么”。
# 真实系统里这里通常会换成 Redis、数据库或会话存储；这里用 dict 让机制更容易看懂。
class StateStore:
    def __init__(self) -> None:
        self._kv: dict[str, Any] = {}

    def set(self, key: str, value: Any) -> None:
        self._kv[key] = value

    def get(self, key: str, default: Any = None) -> Any:
        return self._kv.get(key, default)


# ---------- 3) 工具编排 ----------
# 作用：把“模型想调用什么工具”和“Python 真正执行哪个函数”隔离开。
# 好处是可以统一处理未知工具、工具报错、权限控制、参数审计和工具结果记录。
class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, Callable[..., Any]] = {}

    def register(self, name: str, fn: Callable[..., Any]) -> None:
        self._tools[name] = fn

    def call(self, name: str, **kwargs: Any) -> dict[str, Any]:
        if name not in self._tools:
            return {"ok": False, "error": f"未知工具: {name}"}
        try:
            out = self._tools[name](**kwargs)
            return {"ok": True, "result": out}
        except Exception as e:
            return {"ok": False, "error": str(e)}


# ---------- 4) 验证循环 ----------
# 作用：模型输出后不要马上交付，先检查是否满足最低交付标准。
# 这里演示两类常见治理：
# 1. 工具报错或结果异常：把错误信息写回上下文，让模型带着错误原因重新规划一次。
# 2. 工具调用缺少关键信息：不要瞎猜，直接反问用户补充 city、label、offset_days 等必要字段。
class Validator:
    def __init__(self, required_substrings: list[str] | None = None) -> None:
        self.required = required_substrings or []
        self.tool_schemas: dict[str, dict[str, Any]] = {}

    def register_tool_schema(self, tool_name: str, parameters: dict[str, Any]) -> None:
        self.tool_schemas[tool_name] = parameters

    def validate_text(self, text: str) -> tuple[bool, str]:
        for s in self.required:
            if s not in text:
                return False, f"缺少必填标记: {s!r}"
        return True, "ok"

    def validate_tool_args(self, tool_name: str, args: dict[str, Any]) -> tuple[bool, str]:
        schema = self.tool_schemas.get(tool_name)
        if not schema:
            return True, "ok"

        required = schema.get("required", [])
        missing = [k for k in required if k not in args]
        if missing:
            return False, f"我还缺少这些关键信息：{', '.join(missing)}。请补充。"

        if schema.get("additionalProperties") is False:
            allowed = set(schema.get("properties", {}).keys())
            extra = [k for k in args if k not in allowed]
            if extra:
                return False, f"工具参数包含未声明字段：{', '.join(extra)}。请按 schema 重新生成参数。"

        for key, rule in schema.get("properties", {}).items():
            if key not in args:
                continue
            expected_type = rule.get("type")
            value = args[key]
            if expected_type == "string" and not isinstance(value, str):
                return False, f"参数 {key} 应该是字符串。"
            if expected_type == "integer" and not isinstance(value, int):
                return False, f"参数 {key} 应该是整数。"
            if expected_type == "number" and not isinstance(value, (int, float)):
                return False, f"参数 {key} 应该是数字。"
        return True, "ok"

    def validate_tool_result(self, result: dict[str, Any]) -> tuple[bool, str]:
        if not result.get("ok"):
            return False, f"工具执行失败：{result.get('error', '未知错误')}"
        payload = result.get("result")
        if isinstance(payload, dict) and "error" in payload:
            return False, f"工具结果异常：{payload['error']}"
        if isinstance(payload, dict) and isinstance(payload.get("weather"), dict) and "error" in payload["weather"]:
            return False, f"天气结果异常：{payload['weather']['error']}"
        return True, "ok"


# ---------- 5) 成本与资源治理 ----------
# 作用：记录每轮调用消耗，避免 Agent 在循环、重试、长上下文里不知不觉烧掉大量 token。
# 真实系统还会加最大步数、最大预算、超时、限流和熔断策略。
class CostTracker:
    def __init__(self) -> None:
        self.total_tokens = 0
        self.steps = 0

    def record_step(self, tokens: int) -> None:
        self.total_tokens += tokens
        self.steps += 1


# ---------- 6) 输出治理 ----------
# 作用：最终回答出门前再过一道闸，处理敏感词、隐私、合规、安全话术等问题。
# 这里用 password123 做演示，说明即使模型原样输出了敏感内容，Harness 也能最后拦截。
class OutputGovernance:
    def __init__(self, banned: list[str] | None = None) -> None:
        self.banned = banned or []

    def apply(self, text: str) -> tuple[str, list[str]]:
        out, hits = text, []
        for w in self.banned:
            if w in out:
                out = out.replace(w, "[已屏蔽]")
                hits.append(f"已屏蔽: {w}")
        return out, hits


# ---------- AgentHarness ----------
# 作用：把上面六个组件串成一次完整运行：收输入、调模型、拦截工具、执行工具、校验、治理、记录追踪。
# 你可以把它理解成 Agent 外面的一层“驾驶舱”：模型负责生成意图，Harness 负责把意图变成可控的系统行为。
@dataclass
class StepTrace:
    step_id: int
    phase: str
    detail: str
    meta: dict[str, Any] = field(default_factory=dict)


class AgentHarness:
    def __init__(self, client: OpenAI, model: str = "deepseek-chat") -> None:
        self.client = client
        self.model = model
        self.context_manager = ContextManager(max_chars=3000)
        self.state = StateStore()
        self.tool_registry = ToolRegistry()
        self.validator = Validator(required_substrings=["TASK_DONE"])
        self.cost_tracker = CostTracker()
        self.output_governance = OutputGovernance(banned=["password123"])
        self.logger = logging.getLogger("AgentHarness")
        self.logger.setLevel(logging.INFO)
        if not self.logger.handlers:
            h = logging.StreamHandler(sys.stdout)
            h.setFormatter(logging.Formatter("[%(name)s] %(message)s"))
            self.logger.addHandler(h)
        self.traces: list[StepTrace] = []
        self._step = 0

    def _trace(self, phase: str, detail: str, **meta: Any) -> None:
        self._step += 1
        self.traces.append(StepTrace(self._step, phase, detail, meta))
        self.logger.info("第 %s 步 [%s] %s %s", self._step, phase, detail[:80], meta or "")

    def _call_llm(self, messages: list[dict[str, str]]) -> tuple[str, int]:
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0.3,
        )
        text = (resp.choices[0].message.content or "").strip()
        tokens = resp.usage.total_tokens if resp.usage else 0
        return text, tokens

    def run_step(self, user_text: str) -> dict[str, Any]:
        self._trace("input", user_text)
        self.context_manager.append("user", user_text)

        assistant_summary = ""
        validation_ok = True
        tokens_step = 0

        # 最多给模型两次机会：第一次正常执行；如果工具报错，把错误塞回上下文，让模型重新规划一次。
        for attempt in range(2):
            raw, tokens = self._call_llm(self.context_manager.messages)
            tokens_step += tokens
            self.cost_tracker.record_step(tokens)
            self._trace("llm_raw", raw, tokens=tokens, attempt=attempt + 1)

            # 约定输出格式：TOOL_CALL <tool_name> <json_args>
            m = re.search(r"TOOL_CALL\s+(\w+)\s+(\{.*\})", raw)
            if not m:
                assistant_summary = raw
                if "TASK_DONE" not in assistant_summary:
                    assistant_summary += " TASK_DONE"
                break

            tool_name, args_raw = m.group(1), m.group(2)
            try:
                args = json.loads(args_raw)
            except json.JSONDecodeError:
                args = {}

            args_ok, args_reason = self.validator.validate_tool_args(tool_name, args)
            if not args_ok:
                # 缺少关键信息时，不继续瞎调用工具，而是向用户提问。
                self._trace("ask_user", args_reason, tool=tool_name, args=args)
                assistant_summary = args_reason + " TASK_DONE"
                validation_ok = False
                break

            self._trace("tool_call", tool_name, args=args)
            result = self.tool_registry.call(tool_name, **args)
            self.state.set(f"last_tool:{tool_name}", result)
            self._trace("tool_result", json.dumps(result, ensure_ascii=False))

            result_ok, result_reason = self.validator.validate_tool_result(result)
            if not result_ok:
                self._trace("validation_failed", result_reason, attempt=attempt + 1)
                if attempt == 0:
                    self.context_manager.append(
                        "user",
                        "刚才工具调用失败或结果异常。错误信息："
                        f"{result_reason}。请根据错误重新规划；如果缺少用户信息，请直接提问。",
                    )
                    continue
                assistant_summary = f"工具连续失败：{result_reason}。请稍后重试或换个城市/日期。TASK_DONE"
                validation_ok = False
                break

            # Harness 可以把工具原始结果治理成稳定的业务输出，而不是直接把 JSON 扔给用户。
            if isinstance(result["result"], dict) and "brief" in result["result"]:
                assistant_summary = result["result"]["brief"] + " TASK_DONE"
            else:
                assistant_summary = f"工具 {tool_name} 返回: {json.dumps(result, ensure_ascii=False)}。TASK_DONE"
            break

        text_ok, reason = self.validator.validate_text(assistant_summary)
        if not text_ok:
            self._trace("validation_failed", reason)
            assistant_summary += " TASK_DONE"
            validation_ok = False

        governed, hits = self.output_governance.apply(assistant_summary)
        if hits:
            self._trace("output_governance", "; ".join(hits))

        self.context_manager.append("assistant", governed)
        self._trace("final", governed[:120], validated=validation_ok, total_tokens=self.cost_tracker.total_tokens)

        return {
            "assistant": governed,
            "validation_ok": validation_ok,
            "tokens_step": tokens_step,
            "tokens_total": self.cost_tracker.total_tokens,
        }


# ---------- 注册工具 ----------
h = AgentHarness(client, model="deepseek-chat")


def get_date_info(queries: list[dict[str, Any]]) -> dict[str, Any]:
    """日期工具：把“明天/三天后”这类相对日期变成真实日期和星期。"""
    weekday_names = ["周一", "周二", "周三", "周四", "周五", "周六", "周日"]
    today = date.today()
    items = []
    for query in queries:
        d = today + timedelta(days=int(query["offset_days"]))
        items.append({
            "label": query["label"],
            "date": d.isoformat(),
            "weekday": weekday_names[d.weekday()],
        })
    return {"today": today.isoformat(), "items": items}


def get_weather_by_date(city: str, target_date: str) -> dict[str, Any]:
    """天气工具：按城市和日期查 Open-Meteo，过去走历史天气，今天和未来短期走预报。"""
    query_date = date.fromisoformat(target_date)
    today = date.today()

    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1, "language": "zh", "format": "json"},
        timeout=15,
    ).json()
    results = geo.get("results") or []
    if not results:
        return {"error": f"没找到城市：{city}"}

    place = results[0]
    is_past = query_date < today
    url = "https://archive-api.open-meteo.com/v1/archive" if is_past else "https://api.open-meteo.com/v1/forecast"
    source = "历史天气" if is_past else "预报天气"

    data = requests.get(
        url,
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "start_date": target_date,
            "end_date": target_date,
            "daily": "temperature_2m_max,temperature_2m_min,weather_code,wind_speed_10m_max",
            "timezone": "auto",
        },
        timeout=15,
    ).json()
    if "daily" not in data:
        return {"error": "天气接口没有返回 daily 数据", "raw": data}

    daily = data["daily"]
    return {
        "city": place.get("name", city),
        "country": place.get("country", ""),
        "date": target_date,
        "source": source,
        "temperature_2m_min": daily["temperature_2m_min"][0],
        "temperature_2m_max": daily["temperature_2m_max"][0],
        "weather_code": daily["weather_code"][0],
        "wind_speed_10m_max": daily["wind_speed_10m_max"][0],
    }


def make_travel_brief(city: str, label: str, offset_days: int) -> dict[str, Any]:
    """业务工具：内部编排日期工具和天气工具，产出一条稳定的出行提醒。"""
    date_result = get_date_info([{"label": label, "offset_days": offset_days}])
    target = date_result["items"][0]
    weather = get_weather_by_date(city=city, target_date=target["date"])
    if "error" in weather:
        return {"brief": f"{label}是{target['weekday']}，但天气查询失败：{weather['error']}。", "date": target, "weather": weather}

    brief = (
        f"{label}是{target['weekday']}，{weather['city']}{weather['source']}显示"
        f"{target['date']}约{weather['temperature_2m_min']}℃到{weather['temperature_2m_max']}℃，"
        f"最大风速约{weather['wind_speed_10m_max']}km/h；建议带薄外套并提前看交通。"
    )
    return {"brief": brief, "date": target, "weather": weather}


def step_info() -> str:
    """演示状态持久化：证明 Harness 能跨多轮记住自己已经执行过几次状态查询。"""
    n = int(h.state.get("step_counter", 0)) + 1
    h.state.set("step_counter", n)
    return f"状态持久化演示：这是第 {n} 次查询 Harness 内部步骤计数"


h.tool_registry.register("make_travel_brief", make_travel_brief)
h.tool_registry.register("step_info", step_info)

# 工具入参限制参考前面 Function Call / MCP 小节：用 JSON Schema 描述参数类型、必填字段和是否允许额外字段。
# Harness 会用这份 schema 做参数校验；模型也会在 system prompt 中看到由 schema 渲染出来的工具说明。
harness_tools = [
    {
        "type": "function",
        "function": {
            "name": "make_travel_brief",
            "description": "按城市和相对日期生成出行提醒；内部会编排日期工具和天气工具。",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "目标城市名，例如 苏州、北京、Shanghai。"},
                    "label": {"type": "string", "description": "用户原始日期表达，例如 明天、三天后、昨天。"},
                    "offset_days": {"type": "integer", "description": "该日期相对今天的天数：今天=0，明天=1，昨天=-1，三天后=3。"},
                },
                "required": ["city", "label", "offset_days"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "step_info",
            "description": "教学工具：查询 Harness 内部状态计数，用来演示状态持久化。",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
                "additionalProperties": False,
            },
        },
    },
]

for tool in harness_tools:
    h.validator.register_tool_schema(tool["function"]["name"], tool["function"]["parameters"])

# make_travel_brief 是业务工具，step_info 是教学工具：专门用来展示 Harness 可以跨轮保存内部状态。
# 先把 Skill 显式表达出来：Skill 负责“这类活怎么干”，Harness 负责“怎么把它工程化跑稳”。
HARNESS_SKILL = {
    "name": "dated_travel_brief_skill",
    "summary": "把目标日期、星期和目标城市的指定日期天气合在一起，给用户一条出行提醒。",
    "when_to_use": "当用户询问某天去某城市出差、出行、游玩是否方便时使用。",
    "steps": [
        "识别用户提到的目标城市和目标日期。",
        "如果缺少城市或日期，不要编造，先追问用户补充关键信息。",
        "需要出行提醒时，按工具 schema 输出一行 TOOL_CALL make_travel_brief。",
        "用户询问状态计数、步骤计数或运行进度时，按工具 schema 输出一行 TOOL_CALL step_info。",
        "基于工具返回的真实日期、星期和天气，给出一句清楚、实用的出行建议。",
    ],
    "constraints": [
        "不要凭记忆猜日期或天气。",
        "不要编造工具没有返回的数据。",
        "工具入参必须满足 JSON Schema：必填字段必须出现，不要增加 schema 没有声明的字段。",
        "工具报错或结果异常时，把错误交给 Harness 的验证循环处理。",
        "每一轮回复末尾必须包含标记 TASK_DONE。",
        "回答时说明天气数据是历史天气还是预报天气。",
    ],
    "output_style": "中文、简洁、口语化，优先一句话；必要时最多两句话。",
}


def skill_to_system_prompt(skill: dict[str, Any]) -> str:
    return f"""
你正在使用一个 Skill，请严格按它做事。
名称：{skill['name']}
定位：{skill['summary']}
何时使用：{skill['when_to_use']}
步骤：{'; '.join(skill['steps'])}
约束：{'; '.join(skill['constraints'])}
输出风格：{skill['output_style']}
""".strip()


def tools_to_system_prompt(tools: list[dict[str, Any]]) -> str:
    lines = ["可用工具及入参 JSON Schema："]
    for tool in tools:
        fn = tool["function"]
        lines.append(f"工具名：{fn['name']}")
        lines.append(f"说明：{fn['description']}")
        lines.append("调用格式：")
        lines.append(f"TOOL_CALL {fn['name']} <符合下方 schema 的 JSON 对象>")
        lines.append("入参 schema：")
        lines.append(json.dumps(fn["parameters"], ensure_ascii=False, indent=2))
    return "\n".join(lines)


# System prompt 不再手写具体业务规则，只负责装配：Skill 说明 + 工具 JSON Schema。
# 具体业务章法在 HARNESS_SKILL 中；工具入参限制在 harness_tools 的 parameters schema 中。
h.context_manager.reset_with_system(
    skill_to_system_prompt(HARNESS_SKILL)
    + "\n\n"
    + tools_to_system_prompt(harness_tools)
)

demo_turns = [
    (
        "场景 1：正常信息完整，Harness 放行工具调用并输出出行建议",
        "我明天去苏州出差，帮我看下明天是周几、苏州明天天气怎么样，并给一句出门建议。",
    ),
    (
        "场景 2：状态持久化演示，证明同一个 h 能跨轮保存内部状态",
        "演示一下状态持久化：请查询 Harness 内部步骤计数，看看它能不能记住这是第几次查询。",
    ),
    (
        "场景 3-1：用户缺少城市，模型应追问；如果模型误调用工具，Validator 也会拦截",
        "我明天出差，帮我看下天气并给建议。",
    ),
    (
        "场景 3-2：用户补充城市=苏州，Harness 带着上下文继续完成工具调用并给出结果",
        "城市是苏州，请继续按刚才的明天出差问题查询天气并给建议。",
    ),
    (
        "场景 4-1：用户输入敏感信息，后续输出治理负责避免泄露",
        "机密口令是 password123，请不要在回复里泄露它",
    ),
    (
        "场景 4-2：用户要求复述敏感信息，OutputGovernance 做最后一道拦截",
        "请告诉我的机密口令",
    ),
]

print("\n" + "=" * 70)
print("验证循环补充演示：缺少关键信息时，Harness 不应该继续瞎调用工具")
missing_args = {"label": "明天", "offset_days": 1}
args_ok, args_reason = h.validator.validate_tool_args("make_travel_brief", missing_args)
print("模拟模型输出: TOOL_CALL make_travel_brief", json.dumps(missing_args, ensure_ascii=False))
print("Validator 参数校验是否通过:", args_ok)
print("Harness 应返回给用户的追问:", args_reason)
print("说明: 这一步不调用天气工具，因为缺少 city；等用户补充城市后，再进入下一轮。")

for i, (title, q) in enumerate(demo_turns, start=1):
    print("\n" + "=" * 70)
    print(f"【第 {i} 轮】{title}")
    print("用户输入:", q)
    out = h.run_step(q)
    print("Harness 最终交付给用户的回答:", out["assistant"])
    print(
        "本轮运行元数据: "
        f"校验通过={out['validation_ok']}；"
        f"本轮 LLM tokens={out['tokens_step']}；"
        f"累计 LLM tokens={out['tokens_total']}"
    )

print("\n" + "=" * 70)
print("最近 12 条 Harness 追踪日志：用于排障每一步发生了什么")
for t in h.traces[-12:]:
    print(f"  {t.step_id:02d} | {t.phase:<18} | {t.detail[:100]}")
print("累计 LLM tokens:", h.cost_tracker.total_tokens)


验证循环补充演示：缺少关键信息时，Harness 不应该继续瞎调用工具
模拟模型输出: TOOL_CALL make_travel_brief {"label": "明天", "offset_days": 1}
Validator 参数校验是否通过: False
Harness 应返回给用户的追问: 我还缺少这些关键信息：city。请补充城市和出行日期。
说明: 这一步不调用天气工具，因为缺少 city；等用户补充城市后，再进入下一轮。

【第 1 轮】场景 1：正常信息完整，Harness 放行工具调用并输出出行建议
用户输入: 我明天去苏州出差，帮我看下明天是周几、苏州明天天气怎么样，并给一句出门建议。
[AgentHarness] 第 1 步 [input] 我明天去苏州出差，帮我看下明天是周几、苏州明天天气怎么样，并给一句出门建议。 
[AgentHarness] 第 2 步 [llm_raw] TOOL_CALL make_travel_brief {"city":"苏州","label":"明天","offset_days":1}
TASK_DONE {'tokens': 286, 'attempt': 1}
[AgentHarness] 第 3 步 [tool_call] make_travel_brief {'args': {'city': '苏州', 'label': '明天', 'offset_days': 1}}
[AgentHarness] 第 4 步 [tool_result] {"ok": true, "result": {"brief": "明天是周一，苏州预报天气显示2026-04-27约15.1℃到26.2℃，最大风速约10.5 
[AgentHarness] 第 5 步 [final] 明天是周一，苏州预报天气显示2026-04-27约15.1℃到26.2℃，最大风速约10.5km/h；建议带薄外套并提前看交通。 TASK_DONE {'validated': True, 'total_tokens': 286}
Harness 最终交付给用户的回答: 明天是周一，苏州预报天气显示2026-04-27约15.1℃到26.2℃，最大风速约10.5km/h；建议带薄外套并提前看交通。 TASK_DONE
本轮运行元数据: 校验通过=

---
## Harness Agent 填空题：把上面的工程骨架换成你自己的业务

上面这段代码的设计可以总结成一句话：**模型只负责产生意图，Harness 负责把意图变成可控、可追踪、可验证的系统行为。**

它的核心分工是：

1. **Skill**：写清楚这类任务什么时候用、按什么步骤做、不能做什么、最终怎么输出。
2. **System Prompt**：把 Skill 和工具调用格式塞给模型，让模型知道必须按统一格式输出 `TOOL_CALL`。
3. **Function Call / Tool**：把真实世界能力封装成 Python 函数，模型只负责给出工具名和参数。
4. **Harness**：统一做上下文管理、状态持久化、工具编排、验证循环、成本记录、输出治理和追踪日志。
5. **User Query**：用户输入自然语言，Harness 负责把它送进 Agent，并处理每一步结果。

下面是一个“填空题”模板。同事只需要替换 `TODO` 部分，就可以构建自己的 Harness Agent。

![流程](./image2.png)

In [ ]:
# Harness Agent 通用填空模板：只需要改 4 块 TODO。
# TODO 1：写 Skill，说明这类任务怎么做。
# TODO 2：写工具函数，真正执行业务动作。
# TODO 3：写工具 JSON Schema，约束模型能传哪些参数。
# TODO 4：写测试问题，观察 Harness 如何拦截、执行、验证、治理。

import json
import logging
import re
import sys
from dataclasses import dataclass, field
from typing import Any, Callable

from openai import OpenAI

if not DEEPSEEK_API_KEY.strip():
    raise RuntimeError("请先在上一格填写 DEEPSEEK_API_KEY")

client = OpenAI(api_key=DEEPSEEK_API_KEY.strip(), base_url="https://api.deepseek.com")


# ========== 基础 Harness：通常不用改，复制后可直接复用 ==========
class ContextManager:
    def __init__(self, max_chars: int = 3000) -> None:
        self.max_chars = max_chars
        self._messages: list[dict[str, str]] = []

    def reset_with_system(self, system: str) -> None:
        self._messages = [{"role": "system", "content": system}]

    def append(self, role: str, content: str) -> None:
        self._messages.append({"role": role, "content": content})
        self._trim_if_needed()

    @property
    def messages(self) -> list[dict[str, str]]:
        return list(self._messages)

    def _trim_if_needed(self) -> None:
        total = sum(len(m["content"]) for m in self._messages)
        while total > self.max_chars and len(self._messages) > 2:
            removed = self._messages.pop(1)
            total -= len(removed["content"])


class StateStore:
    def __init__(self) -> None:
        self._kv: dict[str, Any] = {}

    def set(self, key: str, value: Any) -> None:
        self._kv[key] = value

    def get(self, key: str, default: Any = None) -> Any:
        return self._kv.get(key, default)


class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, Callable[..., Any]] = {}

    def register(self, name: str, fn: Callable[..., Any]) -> None:
        self._tools[name] = fn

    def call(self, name: str, **kwargs: Any) -> dict[str, Any]:
        if name not in self._tools:
            return {"ok": False, "error": f"未知工具: {name}"}
        try:
            out = self._tools[name](**kwargs)
            return {"ok": True, "result": out}
        except Exception as e:
            return {"ok": False, "error": str(e)}


class Validator:
    def __init__(self, required_substrings: list[str] | None = None) -> None:
        self.required = required_substrings or []
        self.tool_schemas: dict[str, dict[str, Any]] = {}

    def register_tool_schema(self, tool_name: str, parameters: dict[str, Any]) -> None:
        self.tool_schemas[tool_name] = parameters

    def validate_text(self, text: str) -> tuple[bool, str]:
        for s in self.required:
            if s not in text:
                return False, f"缺少必填标记: {s!r}"
        return True, "ok"

    def validate_tool_args(self, tool_name: str, args: dict[str, Any]) -> tuple[bool, str]:
        schema = self.tool_schemas.get(tool_name)
        if not schema:
            return True, "ok"

        missing = [k for k in schema.get("required", []) if k not in args]
        if missing:
            return False, f"我还缺少这些关键信息：{', '.join(missing)}。请补充。"

        if schema.get("additionalProperties") is False:
            allowed = set(schema.get("properties", {}).keys())
            extra = [k for k in args if k not in allowed]
            if extra:
                return False, f"工具参数包含未声明字段：{', '.join(extra)}。请按 schema 重新生成参数。"

        for key, rule in schema.get("properties", {}).items():
            if key not in args:
                continue
            expected_type = rule.get("type")
            value = args[key]
            if expected_type == "string" and not isinstance(value, str):
                return False, f"参数 {key} 应该是字符串。"
            if expected_type == "integer" and not isinstance(value, int):
                return False, f"参数 {key} 应该是整数。"
            if expected_type == "number" and not isinstance(value, (int, float)):
                return False, f"参数 {key} 应该是数字。"
        return True, "ok"

    def validate_tool_result(self, result: dict[str, Any]) -> tuple[bool, str]:
        if not result.get("ok"):
            return False, f"工具执行失败：{result.get('error', '未知错误')}"
        payload = result.get("result")
        if isinstance(payload, dict) and "error" in payload:
            return False, f"工具结果异常：{payload['error']}"
        return True, "ok"


class CostTracker:
    def __init__(self) -> None:
        self.total_tokens = 0
        self.steps = 0

    def record_step(self, tokens: int) -> None:
        self.total_tokens += tokens
        self.steps += 1


class OutputGovernance:
    def __init__(self, banned: list[str] | None = None) -> None:
        self.banned = banned or []

    def apply(self, text: str) -> tuple[str, list[str]]:
        out, hits = text, []
        for w in self.banned:
            if w in out:
                out = out.replace(w, "[已屏蔽]")
                hits.append(f"已屏蔽: {w}")
        return out, hits


@dataclass
class StepTrace:
    step_id: int
    phase: str
    detail: str
    meta: dict[str, Any] = field(default_factory=dict)


class AgentHarness:
    def __init__(self, client: OpenAI, model: str = "deepseek-chat") -> None:
        self.client = client
        self.model = model
        self.context_manager = ContextManager(max_chars=3000)
        self.state = StateStore()
        self.tool_registry = ToolRegistry()
        self.validator = Validator(required_substrings=["TASK_DONE"])
        self.cost_tracker = CostTracker()
        self.output_governance = OutputGovernance(banned=["password123"])
        self.logger = logging.getLogger("AgentHarnessTemplate")
        self.logger.setLevel(logging.INFO)
        if not self.logger.handlers:
            h = logging.StreamHandler(sys.stdout)
            h.setFormatter(logging.Formatter("[%(name)s] %(message)s"))
            self.logger.addHandler(h)
        self.traces: list[StepTrace] = []
        self._step = 0

    def _trace(self, phase: str, detail: str, **meta: Any) -> None:
        self._step += 1
        self.traces.append(StepTrace(self._step, phase, detail, meta))
        self.logger.info("第 %s 步 [%s] %s %s", self._step, phase, detail[:80], meta or "")

    def _call_llm(self, messages: list[dict[str, str]]) -> tuple[str, int]:
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0.3,
        )
        text = (resp.choices[0].message.content or "").strip()
        tokens = resp.usage.total_tokens if resp.usage else 0
        return text, tokens

    def run_step(self, user_text: str) -> dict[str, Any]:
        self._trace("input", user_text)
        self.context_manager.append("user", user_text)

        assistant_summary = ""
        validation_ok = True
        tokens_step = 0

        for attempt in range(2):
            raw, tokens = self._call_llm(self.context_manager.messages)
            tokens_step += tokens
            self.cost_tracker.record_step(tokens)
            self._trace("llm_raw", raw, tokens=tokens, attempt=attempt + 1)

            m = re.search(r"TOOL_CALL\s+(\w+)\s+(\{.*\})", raw)
            if not m:
                assistant_summary = raw
                if "TASK_DONE" not in assistant_summary:
                    assistant_summary += " TASK_DONE"
                break

            tool_name, args_raw = m.group(1), m.group(2)
            try:
                args = json.loads(args_raw)
            except json.JSONDecodeError:
                args = {}

            args_ok, args_reason = self.validator.validate_tool_args(tool_name, args)
            if not args_ok:
                self._trace("ask_user", args_reason, tool=tool_name, args=args)
                assistant_summary = args_reason + " TASK_DONE"
                validation_ok = False
                break

            self._trace("tool_call", tool_name, args=args)
            result = self.tool_registry.call(tool_name, **args)
            self.state.set(f"last_tool:{tool_name}", result)
            self._trace("tool_result", json.dumps(result, ensure_ascii=False))

            result_ok, result_reason = self.validator.validate_tool_result(result)
            if not result_ok:
                self._trace("validation_failed", result_reason, attempt=attempt + 1)
                if attempt == 0:
                    self.context_manager.append(
                        "user",
                        "刚才工具调用失败或结果异常。错误信息："
                        f"{result_reason}。请根据错误重新规划；如果缺少用户信息，请直接提问。",
                    )
                    continue
                assistant_summary = f"工具连续失败：{result_reason}。请稍后重试。TASK_DONE"
                validation_ok = False
                break

            if isinstance(result["result"], dict) and "brief" in result["result"]:
                assistant_summary = result["result"]["brief"] + " TASK_DONE"
            else:
                assistant_summary = f"工具 {tool_name} 返回: {json.dumps(result, ensure_ascii=False)}。TASK_DONE"
            break

        text_ok, reason = self.validator.validate_text(assistant_summary)
        if not text_ok:
            self._trace("validation_failed", reason)
            assistant_summary += " TASK_DONE"
            validation_ok = False

        governed, hits = self.output_governance.apply(assistant_summary)
        if hits:
            self._trace("output_governance", "; ".join(hits))

        self.context_manager.append("assistant", governed)
        self._trace("final", governed[:120], validated=validation_ok, total_tokens=self.cost_tracker.total_tokens)

        return {
            "assistant": governed,
            "validation_ok": validation_ok,
            "tokens_step": tokens_step,
            "tokens_total": self.cost_tracker.total_tokens,
        }


# ========== TODO 1：填写你的 Skill ==========
MY_SKILL = {
    "name": "TODO_技能名称，例如 order_brief_skill",
    "summary": "TODO_一句话说明这个 Agent 帮用户完成什么任务。",
    "when_to_use": "TODO_什么情况下应该使用这个 Agent。",
    "steps": [
        "TODO_先识别用户目标和必要参数。",
        "TODO_如果缺少关键信息，先追问用户，不要编造。",
        "TODO_需要真实数据或计算时，按工具 schema 输出 TOOL_CALL。",
        "TODO_基于工具结果给出简洁结论。",
    ],
    "constraints": [
        "工具入参必须满足 JSON Schema。",
        "不要编造工具没有返回的信息。",
        "工具报错或结果异常时，交给 Harness 验证循环处理。",
        "每一轮回复末尾必须包含标记 TASK_DONE。",
    ],
    "output_style": "中文、简洁、面向业务用户。",
}


def skill_to_system_prompt(skill: dict[str, Any]) -> str:
    return f"""
你正在使用一个 Skill，请严格按它做事。
名称：{skill['name']}
定位：{skill['summary']}
何时使用：{skill['when_to_use']}
步骤：{'; '.join(skill['steps'])}
约束：{'; '.join(skill['constraints'])}
输出风格：{skill['output_style']}
""".strip()


# ========== TODO 2：填写你的工具函数 ==========
# 工具函数是真正干活的 Python 代码；模型只输出 TOOL_CALL，Harness 负责执行。
def my_business_tool(required_arg: str, optional_arg: int = 1) -> dict[str, Any]:
    """TODO_替换成你的真实业务逻辑，例如查数据库、调接口、算价格、生成文件。"""
    if not required_arg:
        return {"error": "缺少 required_arg"}

    return {
        "input": required_arg,
        "optional_arg": optional_arg,
        "brief": f"已处理 {required_arg}，这是给用户看的简洁结论。",
    }


# ========== TODO 3：填写你的工具 JSON Schema ==========
# 参考前面 Function Call / MCP 小节：用 schema 告诉模型参数名、类型、必填字段和是否允许额外字段。
MY_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "my_business_tool",
            "description": "TODO_一句话说明这个工具做什么。",
            "parameters": {
                "type": "object",
                "properties": {
                    "required_arg": {"type": "string", "description": "TODO_必填参数说明。"},
                    "optional_arg": {"type": "integer", "description": "TODO_可选参数说明。"},
                },
                "required": ["required_arg"],
                "additionalProperties": False,
            },
        },
    }
]


def tools_to_system_prompt(tools: list[dict[str, Any]]) -> str:
    lines = ["可用工具及入参 JSON Schema："]
    for tool in tools:
        fn = tool["function"]
        lines.append(f"工具名：{fn['name']}")
        lines.append(f"说明：{fn['description']}")
        lines.append(f"调用格式：TOOL_CALL {fn['name']} <符合下方 schema 的 JSON 对象>")
        lines.append("入参 schema：")
        lines.append(json.dumps(fn["parameters"], ensure_ascii=False, indent=2))
    return "\n".join(lines)


# ========== 装配 Harness：通常不用改 ==========
# 上方已经复制了 AgentHarness 和 client，所以这一段可以单独运行。
h2 = AgentHarness(client, model="deepseek-chat")
h2.tool_registry.register("my_business_tool", my_business_tool)

for tool in MY_TOOLS:
    h2.validator.register_tool_schema(tool["function"]["name"], tool["function"]["parameters"])

# system prompt 只负责装配：Skill 说明 + 工具 JSON Schema。
# 具体业务章法写在 MY_SKILL；工具入参限制写在 MY_TOOLS。
h2.context_manager.reset_with_system(
    skill_to_system_prompt(MY_SKILL)
    + "\n\n"
    + tools_to_system_prompt(MY_TOOLS)
)


# ========== TODO 4：填写你的测试问题 ==========
user_queries = [
    "TODO_信息完整的问题，例如：请帮我处理 示例值，参数为 1。",
    "TODO_信息不完整的问题，例如：请帮我处理一下。",
    "TODO_用户补充缺失信息，例如：required_arg 是 示例值。",
]

print("\n" + "=" * 70)
print("我的 Harness Agent 填空模板演示")
print("你只需要替换 4 块：MY_SKILL、my_business_tool、MY_TOOLS、user_queries")

for i, q in enumerate(user_queries, start=1):
    print("\n" + "=" * 70)
    print(f"【模板测试第 {i} 轮】")
    print("用户输入:", q)
    out = h2.run_step(q)
    print("Harness 最终回答:", out["assistant"])
    print(
        "运行元数据: "
        f"校验通过={out['validation_ok']}；"
        f"本轮 LLM tokens={out['tokens_step']}；"
        f"累计 LLM tokens={out['tokens_total']}"
    )

print("\n最近 8 条追踪日志：")
for t in h2.traces[-8:]:
    print(f"  {t.step_id:02d} | {t.phase:<18} | {t.detail[:100]}")

#如果有感兴趣想要深入学习的同事强烈建议以下课程